<a href="https://colab.research.google.com/github/iniguezd/cartagenaD0aChatGPT/blob/main/dia4-2/1_smolagents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Herramientas de la calculadora

Como ejemplo vamos a conectar un agente con una *calculadora*. Para ellos vamos a crear una tool por cada operación matemática simple.

In [ ]:
from smolagents import tool

# declaramos las herramientas que va a usar el agente

@tool
def add(a: float, b: float) -> float:
    """
    Adds two numbers together.

    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """
    Subtracts the second number from the first.

    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """
    Multiplies two numbers together.

    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a * b

@tool
def divide(a: float, b: float) -> float | str:
    """
    Divides the first number by the second. Returns an error message if division by zero is attempted.

    Args:
        a (float): The first number.
        b (float): The second number.
    """
    if b == 0:
        return "Error: Division by zero is not allowed."
    return a / b

## Ollama local

Vamos a hacer un ejemplo de calculadora usando smolagents y Ollama.

In [ ]:
from smolagents import LiteLLMModel

# instanciamos el LLM que vamos a utilizar
# en nuestro caso uso de Ollama
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5-coder:7b",
    api_base="http://localhost:11434",
    temperature=0.2,
)

In [ ]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [ ]:
# ejemplo de uso
agent.run("What is 15 multiplied by 3, then subtract 5 and finally divide by 2?")

In [ ]:
# para inspeccionar qué ha pasado internamente
agent.replay(detailed=False)

## Ollama cloud

In [6]:
!pip install smolagents[litellm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.7 MB/s eta 0:00:00


In [7]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/gpt-oss:120b",
    api_base="https://ollama.com",
    temperature=0.2,
    api_key="2aacc506dd4847f09752ba18751b0495.bljCTbHa1btqtpGh8eNTkItP"
)

In [8]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

NameError: name 'add' is not defined

In [9]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

NameError: name 'agent' is not defined

In [ ]:
from smolagents import GradioUI

GradioUI(agent).launch()

## Gemini API

In [ ]:
from smolagents import OpenAIModel

model = OpenAIModel(
    model_id="gemini-2.5-flash",
    # Google Gemini OpenAI-compatible API base URL
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key="<INSERT_YOUR_API_KEY_HERE>",
)

In [ ]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [ ]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

# Open AI API

In [ ]:
from smolagents import OpenAIModel
import os

os.environ["OPENAI_API_KEY"] = "<token>"

model = OpenAIModel(
    model_id="gpt-4o",
    api_base="https://api.openai.com/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

In [ ]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [ ]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

# Ejercicio

Haz un conjunto de herramientas de manera que el LLM interactúe con una base de datos SQLite. Las herramientas deben ser:
- `query_database`. Para ejecutar una SQL `SELECT` y devolver los (100 primeros) resultados.
- `show_tables`. Para listar las tablas de la base de datos.
- `describe_table`. Para describir la estructura de una tabla dada (sus columnas).

Se usa la base de datos de ejemplo Chinook_Sqlite.sqlite.
- Info sobre la base de datos y modelo de datos: https://github.com/lerocha/chinook-database
- Ejemplos de consultas: https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/chinook-queries.sql

El último link usarlo para obtener consultas en lenguaje natural para probar vuestro agente.

In [ ]:
import sqlite3
from typing import Dict, Any

SQLITE_DB_PATH = "Chinook_Sqlite.sqlite"

def get_connection():
    """return connection to the sqlite database"""
    conn = sqlite3.connect(SQLITE_DB_PATH)
    return conn

@tool
def query_database(query: str) -> Dict[str, Any]:
    """Run a SELECT SQL query against the database and return results.

    Args:
        query: SELECT SQL query to execute.

    Returns:
        A dictionary with the query results (rows, columns, etc.).
    """
    if not query.strip().lower().startswith("select"):
        raise ValueError("Only SELECT queries are allowed.")

    cursor = conn.cursor()

    try:

        cursor.execute(query)

        rows = cursor.fetchall()

        columns = [description[0] for description in cursor.description]

        result = {
            "columns": columns,
            "rows": rows,
            "row_count": len(rows)
        }

        return result

    finally:

        conn.close()

@tool
def show_tables() -> Dict[str, Any]:
    """Returns the list of tables in the database."""
    cursor = conn.cursor()

    try:

        query = "SELECT name FROM sqlite_master WHERE type='table';"

        cursor.execute(query)

        rows = cursor.fetchall()

        tables = [row[0] for row in rows]

        result = {
            "tables": tables,
            "count": len(tables)
        }

        return result

    finally:

        conn.close()

@tool
def describe_table(table_name: str) -> Dict[str, Any]:
    """Returns the schema of the specified table.
    Args:
        table_name: Name of the table to describe.
    Returns:
        A dictionary with the table schema information.
    """
    pass

In [ ]:
model = "Qwen/Qwen3.5-4B"
agent = ToolCallingAgent(model=model, tools=[query_database, show_tables, describe_table],)
agent.run("Who is the customer that spent the most money? Give me the name")

In [ ]:
from smolagents import GradioUI

# si queréis jugar con el agente en una interfaz gráfica
GradioUI(agent).launch()